In [1]:
import numpy as np
import pandas as pd
from sklearn import *
import lightgbm as lgb

train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')
sub = pd.read_csv('../input/sample_submission.csv')
print(train.shape, test.shape, sub.shape)

train['atom'] = train['type'].map(lambda x: str(x)[3])
test['atom'] = test['type'].map(lambda x: str(x)[3])

lbl = preprocessing.LabelEncoder()
for i in range(4):
    train['type'+str(i)] = lbl.fit_transform(train['type'].map(lambda x: str(x)[i]))
    test['type'+str(i)] = lbl.transform(test['type'].map(lambda x: str(x)[i]))

structures = pd.read_csv('../input/structures.csv').rename(columns={'atom_index':'atom_index_0', 'x':'x0', 'y':'y0', 'z':'z0'})
train = pd.merge(train, structures, how='left', on=['molecule_name', 'atom_index_0', 'atom'])
test = pd.merge(test, structures, how='left', on=['molecule_name', 'atom_index_0', 'atom'])
del structures

structures = pd.read_csv('../input/structures.csv').rename(columns={'atom_index':'atom_index_1', 'x':'x1', 'y':'y1', 'z':'z1'})
train = pd.merge(train, structures, how='left', on=['molecule_name', 'atom_index_1', 'atom'])
test = pd.merge(test, structures, how='left', on=['molecule_name', 'atom_index_1', 'atom'])
del structures
print(train.shape, test.shape, sub.shape)


(4191263, 6) (467813, 5) (467813, 2)
(4191263, 17) (467813, 16) (467813, 2)


In [2]:
#https://www.kaggle.com/artgor/molecular-properties-eda-and-models
train_p0 = train[['x0', 'y0', 'z0']].fillna(0).values
train_p1 = train[['x1', 'y1', 'z1']].fillna(0).values
test_p0 = test[['x0', 'y0', 'z0']].fillna(0).values
test_p1 = test[['x1', 'y1', 'z1']].fillna(0).values

train['dist'] = np.linalg.norm(train_p0 - train_p1, axis=1)
test['dist'] = np.linalg.norm(test_p0 - test_p1, axis=1)

train['dist_to_type_mean'] = train['dist'] / train.groupby('type')['dist'].transform('mean')
test['dist_to_type_mean'] = test['dist'] / test.groupby('type')['dist'].transform('mean')


In [3]:
col = [c for c in train.columns if c not in ['id', 'molecule_name', 'scalar_coupling_constant', 'type', 'atom']]

def lgb_lmae(preds, dtrain):
    labels = dtrain.get_label()
    score = np.log(metrics.mean_absolute_error(labels, preds))
    return 'lmae', score, False

params = {'boosting_type': 'gbdt', 'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.2, 'num_leaves': 64}

x1, x2, y1, y2 = model_selection.train_test_split(train[col], train['scalar_coupling_constant'], test_size=0.2, random_state=99)
model = lgb.train(params, lgb.Dataset(x1, label=y1), 400, lgb.Dataset(x2, label=y2),early_stopping_rounds=20, verbose_eval=20, feval=lgb_lmae)
test['scalar_coupling_constant']  = model.predict(test[col], num_iteration=model.best_iteration)
test[['id', 'scalar_coupling_constant']].to_csv('submission.csv', float_format='%.9f', index=False)     


TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'